# Do LLMs Split Couples' Chores Fairly? — Colab runner

**Before you start:** Runtime → Change runtime type → **L4 GPU**. Add your Hugging Face token in the key icon (Secrets) on the left as `HF_TOKEN`, with notebook access on.

All outputs are written to Google Drive, so nothing is lost if the runtime disconnects. **After a disconnect:** reconnect, run cells 1–5 again, then rerun the run cell you were on. Finished prompts are skipped.

### 1. Check the GPU

In [ ]:
!nvidia-smi

### 2. Mount Google Drive (outputs go here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/chores_llm_outputs'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

### 3. Get the code

In [ ]:
REPO = 'https://github.com/keeljimin/do_LLMs_split_couples_chores_fairly.git'
if not os.path.exists('/content/repo'):
    !git clone {REPO} /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!git log -1 --oneline

### 4. Install vLLM (a few minutes)

In [ ]:
!pip install -q -r requirements.txt
import vllm; print('vllm', vllm.__version__)

### 5. Hugging Face login

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!huggingface-cli whoami

### 6. Build the prompts
Check that the example prompt matches the pilot prompt word for word.

In [ ]:
!python make_prompts.py

### 7. Gemma sample (240 prompts)
If you get an out-of-memory error, add `--mm-off` (and if needed `--max-model-len 8192`).

In [ ]:
!python run.py --model gemma --sample --outdir {OUT}

### 8. Check the sample
Copy the printed summary and compare the WFH shares with the pilot (wife WFH 55%, husband WFH 61%, baseline 52%).

In [ ]:
!python validate.py {OUT}/outputs_gemma.jsonl --outdir {OUT}

### 9. Full run: Gemma
Resumes after the sample. Keep this tab open: the runtime stops if the browser closes or the session cap is reached.

In [ ]:
!python run.py --model gemma --outdir {OUT}

### 10. Full run: Qwen (instruct mode)
Qwen in bf16 is tight on the L4; if it runs out of memory, add `--mm-off --max-model-len 8192`.

In [ ]:
!python run.py --model qwen --outdir {OUT}

### 11. Validate everything

In [ ]:
!python validate.py {OUT}/outputs_gemma.jsonl {OUT}/outputs_qwen.jsonl --outdir {OUT}